# M2 · Process collections and manage changes

Collections organize many values; operations select, combine, and change owned state. Trace each change before running it.

**Start:** Use a Python 3 kernel; these cells use only the standard library. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

## Exploration 1 — build and observe the mechanism

### 1. Choose collection shapes

Different collections serve different operations. The catalog supports lookup; the cart associates IDs with quantities.

**Before running:** Predict the result or state change.

In [ ]:
products = {
    "1": {"name": "USB-C Cable", "price_paise": 49900},
    "3": {"name": "Water Bottle", "price_paise": 89900},
}
cart = {"1": 2, "3": 1}
selected_ids = ["1", "1", "3"]
print(products["1"]["name"], selected_ids[0])
print(set(selected_ids), ("1", 2))


**Change and explain:** Explain why the set cannot retain two cable selections.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The list preserves repeated IDs; the set represents uniqueness; the tuple groups two values.

</details>

### 2. Traverse keys and records deliberately

Dictionary iteration supplies keys. Choose values or items when the operation needs the associated records.

**Before running:** Predict the result or state change.

In [ ]:
for product_id, product in products.items():
    print(product_id, product["name"])
print("999" in products)
print(products.get("999"))


**Change and explain:** Replace items() with values() and adapt the loop variables.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Both known products are displayed; missing membership is False and get returns None.

</details>

### 3. Accumulate contributions

Initialize an accumulator once and combine each member’s contribution. The initial value defines the empty result.

**Before running:** Predict the result or state change.

In [ ]:
def subtotal(cart):
    total_paise = 0
    for product_id, quantity in cart.items():
        total_paise += products[product_id]["price_paise"] * quantity
    return total_paise

print(subtotal(cart))
print(subtotal({}))


**Change and explain:** Change the cable quantity to 3 and rerun.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The original cart totals 189700; an empty cart totals zero.

</details>

### 4. Select and order records

Selection and sorting derive a result without rewriting the source. A short comprehension expresses a simple selection.

**Before running:** Predict the result or state change.

In [ ]:
matches = []
for product in products.values():
    if "cable" in product["name"].casefold():
        matches.append(product["name"])
print(matches)
print([p["name"] for p in products.values() if p["price_paise"] < 60000])
print(sorted(products.values(), key=lambda p: p["price_paise"]))


**Change and explain:** Raise the bound to 100000; explain which collection changes.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only the cable matches the first two selections; sorting produces price order.

</details>

### 5. Group repeated selections

Accumulation can produce a collection as well as a number. Positions, groups, and uniqueness answer different questions.

**Before running:** Predict the result or state change.

In [ ]:
quantities = {}
for position, product_id in enumerate(selected_ids, start=1):
    quantities[product_id] = quantities.get(product_id, 0) + 1
    print(position, product_id)
print(quantities)
print(list(range(3)))


**Change and explain:** Try an empty selected_ids list.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Cable quantity 2 and bottle quantity 1; range excludes its upper bound.

</details>

### 6. Repeat until a condition ends the interaction

A command loop may have an unknown length. Continue skips one iteration; break ends the loop.

**Before running:** Predict the result or state change.

In [ ]:
commands = iter(["", "browse", "quit"])
while True:
    command = next(commands)
    if not command:
        continue
    if command == "quit":
        break
    print("Command:", command)


**Change and explain:** Insert "cart" before "quit".

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only browse is handled; the empty command is skipped and quit ends the loop.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Distinguish an absent bound from zero

An interface must assign meaning to omission. A valid zero bound must not disappear through a truthiness check.

**Before running:** Predict the result or state change.

In [ ]:
def within_budget(price_paise: int, *, maximum_paise=None) -> bool:
    """Check an optional upper price bound without mutating data."""
    return maximum_paise is None or price_paise <= maximum_paise

print(within_budget(49900))
print(within_budget(49900, maximum_paise=0))
print(within_budget(0, maximum_paise=0))


**Change and explain:** Explain why testing only if maximum_paise would be wrong.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

No bound accepts the cable; a zero bound rejects it but accepts a free product.

</details>

### 8. Trace shared nested state

Another name shares an object. A shallow copy creates a new outer container but retains references to nested objects.

**Before running:** Predict the result or state change.

In [ ]:
items = [{"product_id": "1", "quantity": 2}]
alias = items
outer_copy = items.copy()
outer_copy[0]["quantity"] = 5
print(items is alias, items is outer_copy)
print(items[0]["quantity"], outer_copy[0]["quantity"])


**Change and explain:** Append a new item to outer_copy; which outer list changes?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The lists have different identities but both see the shared inner quantity 5.

</details>

### 9. Observe a default with the wrong lifetime

A default is created when the definition executes. Reusing mutable default state can connect independent calls.

**Before running:** Predict the result or state change.

In [ ]:
def remember(product_id, selected_ids=[]):
    selected_ids.append(product_id)
    return selected_ids

print(remember("1"))
print(remember("3"))


**Change and explain:** Rerun this entire definition cell; explain why that creates a new default.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The second call sees both IDs, because both calls used the same list.

</details>

### 10. Construct fresh state for each omitted cart

Create changing state at the lifetime of its owner. Explicitly supplied state can still be changed deliberately.

**Before running:** Predict the result or state change.

In [ ]:
def remember(product_id, selected_ids=None):
    if selected_ids is None:
        selected_ids = []
    selected_ids.append(product_id)
    return selected_ids

print(remember("1"))
print(remember("3"))


**Change and explain:** Supply the same list to both calls and predict the result.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Omitted lists are independent: ["1"] and ["3"].

</details>

### 11. Reject before changing the cart

Validate an operation’s prerequisites before mutation. A rejection should preserve the existing state.

**Before running:** Predict the result or state change.

In [ ]:
def add_item(cart, product_id, quantity):
    if type(quantity) is not int or quantity <= 0:
        raise ValueError("Quantity must be a positive integer")
    if product_id not in products:
        raise KeyError(product_id)
    cart[product_id] = cart.get(product_id, 0) + quantity

fresh_cart = {}
add_item(fresh_cart, "1", 2)
print(fresh_cart)


**Change and explain:** What must remain unchanged if adding quantity zero fails?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The fresh cart contains two cables; no global cart is implicitly changed.

</details>

### 12. Handle an expected input failure narrowly

Catch failures you understand at the interaction boundary. Leave unrelated defects visible.

**Before running:** Predict the result or state change.

In [ ]:
for text in ["0", "two", "1"]:
    try:
        add_item(fresh_cart, "1", int(text))
    except ValueError as error:
        print("Rejected:", text, str(error))
print(fresh_cart)


**Change and explain:** Try True directly as a quantity; explain the exact integer check.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Zero and nonnumeric text are rejected; the accepted unit brings the cable quantity to three.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M2 lab guide](../../projects/nano-zon/labs/m02/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.